In [1]:
import pyvisa as visa
import numpy as np
import matplotlib.pyplot as plt
import instrumentlib as inst

In [2]:
def set_V_read_I(keithley, V, I_c=1E-1):
    
    keithley.write(f"SOUR:VOLT:MODE FIXED\n SOUR:VOLT {V}")
    keithley.write(f"SENS:CURR:PROT {I_c}")
    keithley.write("TRIG:COUN 1")
    keithley.write("FORM:ELEM CURR")
    keithley.write("OUTP ON")
    keithley.write("READ?")
    I_read = keithley.read()
    
    return float(I_read)

def reset(keithley):
    keithley.write("*RST")
    set_V_read_I(keithley, 0, 0.1)

def set_frequency_GHz(srs, f):
    
    srs.write(f"FREQ{f}GHz")

def set_amp_dBm(srs, a):
    
    f = float(srs.query("FREQ? GHz"))
    if(f > 6.075):
        srs.write(f"AMPH {a}")
    else : 
        srs.write(f"AMPL {a}")



def get_frequency_GHz(srs):
    
    f = srs.query("FREQ?GHz")
    
    return float(f)

def get_amp_dBm(srs):
    
    f = float(srs.query("FREQ? GHz"))
    if(f > 6.075):
        a = srs.query("AMPH?")
    else : 
        a = srs.query("AMPL?")
    
    return float(a)

def set_modulation_off(srs):
    
    srs.write("MODL 0")

def set_pumpAmp_FlxVoltage(srs, keithley, a, V, I_c=1E-1):
    
    set_amp_dBm(srs, a)
    I_read = set_V_read_I(keithley, V, I_c=1E-1)
    a_read = get_amp_dBm(srs)
    
    return I_read, a_read

def rampdown_V(keithley, V_init, V_fin, step=0.01, t_wait=1, verbose=False):
    
    import time
    step = np.sign(V_fin - V_init)*np.abs(step)
    V_list = np.round(np.arange(V_init, V_fin + step, step),3) 
    
    for V in V_list:
        I_read = set_V_read_I(keithley, V)
        time.sleep(t_wait)
        if verbose:
            print(I_read, V)
    return I_read
        
        

In [3]:
rm = visa.ResourceManager()
rm.list_resources()

('GPIB0::17::INSTR',)

In [4]:
# keithley1 = rm.open_resource('GPIB0::24::INSTR') # Old 
keithley2 = rm.open_resource('GPIB0::17::INSTR') # New
# srs2 = rm.open_resource("TCPIP0::192.168.0.162::inst0::INSTR")
# srs4 = rm.open_resource("TCPIP0::192.168.0.160::inst0::INSTR")
# print(keithley1.query('*IDN?'))
# print(keithley2.query('*IDN?'))
# print(srs2.query('*IDN?'))
# print(srs4.query('*IDN?'))

In [21]:
V_init, V_fin = 10 ,0
rampdown_V(keithley2, V_init, V_fin, step=0.2, t_wait=0.3, verbose=True)

0.004649695 10.0
0.004553791 9.8
0.004460795 9.6
0.004367647 9.4
0.004274731 9.2
0.004181853 9.0
0.004088658 8.8
0.003995778 8.6
0.003902789 8.4
0.003809885 8.2
0.00371667 8.0
0.003623812 7.8
0.003530864 7.6
0.003437776 7.4
0.003344831 7.2
0.003251903 7.0
0.003158943 6.8
0.00306583 6.6
0.00297288 6.4
0.002879963 6.2
0.002786794 6.0
0.002693846 5.8
0.002600973 5.6
0.002507998 5.4
0.002414854 5.2
0.002321912 5.0
0.00222896 4.8
0.002135832 4.6
0.002042927 4.4
0.001949994 4.2
0.001857067 4.0
0.001763897 3.8
0.001670914 3.6
0.001578025 3.4
0.001485057 3.2
0.001391879 3.0
0.001299023 2.8
0.001205988 2.6
0.001112841 2.4
0.001019994 2.2
0.0009299586 2.0
0.0008340272 1.8
0.0007410068 1.6
0.0006480039 1.4
0.0005550007 1.2
0.0004620212 1.0
0.0003689538 0.8
0.0002760381 0.6
0.0001829939 0.4
9.295296e-05 0.2
-8.085883e-09 0.0


-8.085883e-09

In [ ]:
# set_modulation_off(srs2)
set_modulation_off(srs4)

In [ ]:
rr1_LO = 7.5
rr2_LO = 7.00005
set_frequency_GHz(srs4, rr1_LO)
# set_frequency_GHz(srs2, rr2_LO)
# f2 = get_frequency_GHz(srs2)
f4 = get_frequency_GHz(srs4)

# print(f"Frequency of SRS2 = {f2} GHz")
print(f"Frequency of SRS4 = {f4} GHz")

In [ ]:
srs4.write("ENBH 1")
# srs2.write("ENBH 0")

In [ ]:
#srs2.write("DISP 8")
srs4.write("DISP 8")

# Keithley 2 and SRS 4 #
# Input 18 Flux Line 1 #

In [18]:
keithley = keithley2
V = 7.5#  7.85 V first bias 7.15 for dual tone  #8.8 V For single tone
set_V_read_I(keithley, V, I_c=1E-1)
#rampdown_V(keithley,-3.92,-4.33, step=0.01, t_wait=2)

0.003770527

In [ ]:
#set_V_read_I(keithley2, -0.1)
srs = srs4
keithley = keithley2
amp = 8.98
V = 1.39
set_pumpAmp_FlxVoltage(srs, keithley, amp, V)

#  Keithley 1 and SRS 2 #
# Input 17 Flux Line 2 #

In [ ]:
#set_V_read_I(keithley2, -0.1)
# srs = srs2
# keithley =keithley1
srs = srs4
keithley = keithley2
amp = 2.91
V =   -0.82
set_pumpAmp_FlxVoltage(srs, keithley, amp, V)

In [ ]:
get_amp_dBm(srs4)

In [ ]:
get_frequency_GHz(srs2)

In [ ]:
rampdown_V(keithley2, 0.5, 0, step = 0.01)

In [ ]:
srs2.close()
srs4.close()
keithley1.close()
keithley2.close()

In [ ]:
rm.list_resources()

In [ ]:
import time
import matplotlib.pyplot as plt
keithley = keithley1


I = []
for i in range(1000):
    keithley.write("FORM:ELEM CURR")
    keithley.write("OUTP ON")
    keithley.write("READ?")
    I_read = float(keithley.read())
    print(I_read)
    I.append(1e3*I_read)
    time.sleep(5)
    
    plt.plot(I)
    plt.grid()
    plt.ylabel("Current (mA)")
    #plt.show()

In [ ]:
set_amp_dBm(srs4, -10)

In [ ]:
fr = 7.633
BW = 7.838 - 7.431

print("Quality factor:{}".format(np.around(fr/BW, 2)))

In [ ]:
fr = 7.682
BW = 7.960 - 7.476

print("Quality factor:{}".format(np.around(fr/BW, 2)))